# Hybrid Neural Collaborative Filtering, with 
## Imports & Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import optuna
from pathlib import Path
from tqdm.auto import tqdm
import random
import os

# Ensure deterministic behavior where possible
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define Paths
DATA_DIR = Path.cwd().parent / "data"
SUB_DIR = Path.cwd().parent / "submissions"

INTERACTIONS_PATH = DATA_DIR / "interactions_train.csv"
ITEMS_PATH = DATA_DIR / "items.csv"
ENRICHED_ITEMS_PATH = DATA_DIR / "final_enriched_items.csv" 
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

## Data Loading and Merging

In [ ]:
interactions = pd.read_csv(INTERACTIONS_PATH)

items = pd.read_csv(ITEMS_PATH)

# Load enriched data (if available), else just use items
if ENRICHED_ITEMS_PATH.exists():
    enriched_df = pd.read_csv(ENRICHED_ITEMS_PATH)
    # Merge on item ID ("i")
    # If the API script already merged them, you can skip merging, but this is safe:
    if "genres" in enriched_df.columns and "genres" not in items.columns:
        df = items.merge(enriched_df[["item_id", "genres", "summary", "page_count", "average_rating"]], 
                         left_on="i", right_on="item_id", how="left")
    else:
        df = enriched_df.copy()
else:
    print("Warning: final_enriched_items.csv not found. Falling back to basic items.csv")
    df = items.copy()

df.fillna("", inplace=True)

n_users = interactions["u"].nunique()
n_items = items["i"].nunique()
print(f"Users: {n_users}, Items: {n_items}, Interactions: {len(interactions)}")

## Feature Extraction (Text & Numerical) - Text Processing & Embeddings

In [ ]:
MODEL_CHOICE = "e5" 

model_mapping = {
    "minilm": "paraphrase-multilingual-MiniLM-L12-v2",
    "e5": "intfloat/multilingual-e5-base",
    "gemma": "google/embeddinggemma-300m"
}

print(f"Preparing Text Features using {model_mapping[MODEL_CHOICE]}")
rich_texts = []

for _, row in df.iterrows():
    title = str(row.get('Title', row.get('title', ''))).strip()
    author = str(row.get('Author', row.get('author', ''))).strip()
    subjects = str(row.get('Subjects', row.get('subjects', ''))).strip() 
    genres = str(row.get('genres', '')).strip()
    summary = str(row.get('summary', '')).strip()
    
    parts = []
    if title: parts.append(f"Title: {title}.")
    if author: parts.append(f"Author: {author}.")
    if subjects: parts.append(f"Subjects: {subjects}.")
    if genres: parts.append(f"Genres: {genres}.")
    if summary: parts.append(f"Summary: {summary}.")
    
    full_text = " ".join(parts) if parts else "Unknown book."
    
    # E5 models require a prefix for document indexing
    if MODEL_CHOICE == "e5":
        full_text = "passage: " + full_text
        
    rich_texts.append(full_text)

# Encode Texts
embedder = SentenceTransformer(model_mapping[MODEL_CHOICE], device=device)
embeddings = embedder.encode(rich_texts, show_progress_bar=True, convert_to_numpy=True)
text_embed_dim = embeddings.shape[1] # Automatically grabs 384 or 768 based on model

# --- Numerical Features ---
print("Preparing Numerical Features")
if "page_count" in df.columns and "average_rating" in df.columns:
    # Replace empty strings mapped during fillna with NaN for imputation
    num_df = df[["page_count", "average_rating"]].replace("", np.nan)
else:
    # Dummy numerical features if not found
    num_df = pd.DataFrame({"page_count": [np.nan]*len(df), "average_rating": [np.nan]*len(df)})

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
num_features_scaled = scaler.fit_transform(imputer.fit_transform(num_df))
num_numerical_cols = num_features_scaled.shape[1]

# --- Map to Tensors using Item IDs ---
max_item_id = int(df["i"].max())
item_text_tensor = torch.zeros((max_item_id + 1, text_embed_dim), dtype=torch.float32)
item_num_tensor = torch.zeros((max_item_id + 1, num_numerical_cols), dtype=torch.float32)

item_ids = df["i"].values.astype(int)
item_text_tensor[item_ids] = torch.tensor(embeddings, dtype=torch.float32)
item_num_tensor[item_ids] = torch.tensor(num_features_scaled, dtype=torch.float32)

# Move to device to keep them ready for training
item_text_tensor = item_text_tensor.to(device)
item_num_tensor = item_num_tensor.to(device)
print(f"Text Tensor Shape: {item_text_tensor.shape}, Num Tensor Shape: {item_num_tensor.shape}")

## PyTorch Dataset & Model Definition

In [ ]:
class BookInteractionDataset(Dataset):
    def __init__(self, df, num_items, num_negatives=4):
        self.users = df["u"].values
        self.items = df["i"].values
        self.num_items = num_items
        self.num_negatives = num_negatives
        self.user_positives = df.groupby("u")["i"].apply(set).to_dict()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        user = self.users[idx]
        pos_item = self.items[idx]
        
        neg_items = []
        for _ in range(self.num_negatives):
            neg_item = random.randint(0, self.num_items - 1)
            while neg_item in self.user_positives[user]:
                neg_item = random.randint(0, self.num_items - 1)
            neg_items.append(neg_item)
            
        return torch.tensor(user, dtype=torch.long), \
               torch.tensor(pos_item, dtype=torch.long), \
               torch.tensor(neg_items, dtype=torch.long)

class HybridBookRecommender(nn.Module):
    def __init__(self, num_users, num_items, embed_dim, text_embed_dim, num_numerical_features, dropout_rate=0.2):
        super(HybridBookRecommender, self).__init__()
        
        self.user_embedding = nn.Embedding(num_embeddings=num_users, embedding_dim=embed_dim)
        self.item_embedding = nn.Embedding(num_embeddings=num_items, embedding_dim=embed_dim)
        
        mlp_input_dim = embed_dim + embed_dim + text_embed_dim + num_numerical_features
        
        self.mlp = nn.Sequential(
            nn.Linear(mlp_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.output_layer = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_idx, item_idx, text_features, num_features):
        u_embed = self.user_embedding(user_idx)
        i_embed = self.item_embedding(item_idx)
        
        concat_features = torch.cat([u_embed, i_embed, text_features, num_features], dim=-1)
        mlp_out = self.mlp(concat_features)
        return self.sigmoid(self.output_layer(mlp_out)).squeeze()

## Optuna Hyperparameter Tuning

In [ ]:
# 1. Temporal Split (80% Train, 20% Val) to avoid data leakage
interactions.sort_values("t", inplace=True)
split_idx = int(len(interactions) * 0.8)
train_df = interactions.iloc[:split_idx]
val_df = interactions.iloc[split_idx:]

train_dataset = BookInteractionDataset(train_df, num_items=max_item_id + 1, num_negatives=4)
val_dataset = BookInteractionDataset(val_df, num_items=max_item_id + 1, num_negatives=4)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, num_workers=0)

def objective(trial):
    embed_dim = trial.suggest_categorical("embed_dim", [32, 64, 128])
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    
    model = HybridBookRecommender(
        num_users=interactions["u"].max() + 1, 
        num_items=max_item_id + 1, 
        embed_dim=embed_dim, 
        text_embed_dim=text_embed_dim, 
        num_numerical_features=num_numerical_cols,
        dropout_rate=dropout_rate
    ).to(device)
    
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    epochs_for_tuning = 3 
    
    for epoch in range(epochs_for_tuning):
        model.train()
        for users, pos_items, neg_items in train_loader:
            users, pos_items, neg_items = users.to(device), pos_items.to(device), neg_items.to(device)
            
            batch_users = users.repeat_interleave(1 + train_dataset.num_negatives)
            batch_items = torch.cat([pos_items.unsqueeze(1), neg_items], dim=1).flatten()
            
            pos_labels = torch.ones_like(pos_items, dtype=torch.float32)
            neg_labels = torch.zeros_like(neg_items, dtype=torch.float32)
            batch_labels = torch.cat([pos_labels.unsqueeze(1), neg_labels], dim=1).flatten().to(device)
            
            batch_text_features = item_text_tensor[batch_items]
            batch_num_features = item_num_tensor[batch_items]
            
            optimizer.zero_grad()
            predictions = model(batch_users, batch_items, batch_text_features, batch_num_features)
            loss = criterion(predictions, batch_labels)
            loss.backward()
            optimizer.step()
            
    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for users, pos_items, neg_items in val_loader:
            users, pos_items, neg_items = users.to(device), pos_items.to(device), neg_items.to(device)
            batch_users = users.repeat_interleave(1 + val_dataset.num_negatives)
            batch_items = torch.cat([pos_items.unsqueeze(1), neg_items], dim=1).flatten()
            
            pos_labels = torch.ones_like(pos_items, dtype=torch.float32)
            neg_labels = torch.zeros_like(neg_items, dtype=torch.float32)
            batch_labels = torch.cat([pos_labels.unsqueeze(1), neg_labels], dim=1).flatten().to(device)
            
            batch_text_features = item_text_tensor[batch_items]
            batch_num_features = item_num_tensor[batch_items]
            
            predictions = model(batch_users, batch_items, batch_text_features, batch_num_features)
            loss = criterion(predictions, batch_labels)
            val_loss += loss.item()
            
    return val_loss / len(val_loader)

# Run Optuna Study (Set n_trials higher if you have more time!)
study = optuna.create_study(direction="minimize", study_name="Hybrid_Recommender_Tuning")
study.optimize(objective, n_trials=10)

best_params = study.best_trial.params
print(f"Best Validation Loss: {study.best_trial.value:.4f}")
print("Best Hyperparameters:", best_params)

## Final Training

In [ ]:
# Combine train/val back into a single full dataset
full_dataset = BookInteractionDataset(interactions, num_items=max_item_id + 1, num_negatives=4)
full_loader = DataLoader(full_dataset, batch_size=512, shuffle=True, num_workers=0)

final_model = HybridBookRecommender(
    num_users=interactions["u"].max() + 1, 
    num_items=max_item_id + 1, 
    embed_dim=best_params["embed_dim"], 
    text_embed_dim=text_embed_dim, 
    num_numerical_features=num_numerical_cols,
    dropout_rate=best_params["dropout_rate"]
).to(device)

criterion = nn.BCELoss()
optimizer = optim.AdamW(final_model.parameters(), lr=best_params["lr"], weight_decay=best_params["weight_decay"])

# Train longer on the full set
num_epochs = 12 
final_model.train()

for epoch in range(num_epochs):
    total_loss = 0.0
    loop = tqdm(full_loader, leave=False, desc=f"Final Epoch {epoch+1}/{num_epochs}")
    for users, pos_items, neg_items in loop:
        users, pos_items, neg_items = users.to(device), pos_items.to(device), neg_items.to(device)
        
        batch_users = users.repeat_interleave(1 + full_dataset.num_negatives)
        batch_items = torch.cat([pos_items.unsqueeze(1), neg_items], dim=1).flatten()
        
        pos_labels = torch.ones_like(pos_items, dtype=torch.float32)
        neg_labels = torch.zeros_like(neg_items, dtype=torch.float32)
        batch_labels = torch.cat([pos_labels.unsqueeze(1), neg_labels], dim=1).flatten().to(device)
        
        batch_text_features = item_text_tensor[batch_items]
        batch_num_features = item_num_tensor[batch_items]
        
        optimizer.zero_grad()
        predictions = final_model(batch_users, batch_items, batch_text_features, batch_num_features)
        
        loss = criterion(predictions, batch_labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

## Submission Generation

In [ ]:
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

# Dictionary of {user_id: set_of_items_read}
user_histories = interactions.groupby("u")["i"].apply(set).to_dict()

predictions_list = []
final_model.eval()

with torch.no_grad():
    for user in tqdm(sample_submission["user_id"]):
        read_items = user_histories.get(user, set())
        
        # Filter to only score unread items (0 to max_item_id)
        unread_items = [i for i in range(max_item_id + 1) if i not in read_items]
        unread_items_tensor = torch.tensor(unread_items, dtype=torch.long).to(device)
        user_tensor = torch.tensor([user] * len(unread_items), dtype=torch.long).to(device)
        
        unread_text_features = item_text_tensor[unread_items_tensor]
        unread_num_features = item_num_tensor[unread_items_tensor]
        
        # Score all unread items
        scores = final_model(user_tensor, unread_items_tensor, unread_text_features, unread_num_features)
        
        # Get indices of top 10 scores
        # scores correspond to the unread_items array
        top_10_local_idx = torch.topk(scores, min(10, len(scores))).indices
        top_10_item_ids = unread_items_tensor[top_10_local_idx].cpu().numpy()
        
        # Format for submission (space separated)
        predictions_list.append(" ".join(map(str, top_10_item_ids)))

sample_submission["recommendation"] = predictions_list
out_path = SUB_DIR / f"submission_hybrid_ncf_{MODEL_CHOICE}.csv"
sample_submission.to_csv(out_path, index=False)

print(f"Success! Final submission saved to: {out_path}")